# 🛰️ OrbitalDelta — GPU Training on Google Colab

Run this notebook in **Google Colab** with a GPU enabled (Runtime → Change runtime type → **T4 GPU**).  
Cells must be run **top to bottom** in order. Everything is automated — no manual downloads needed.

## 1. Clone Repo & Install Dependencies

In [ ]:
!git clone https://github.com/MihirJayswal/OrbitalDelta.git
%cd OrbitalDelta

!pip install -q torchmetrics albumentations rasterio geopandas fiona shapely pyproj
!pip install -q opencv-python-headless tqdm pyyaml scikit-learn matplotlib seaborn
!pip install -q fastapi uvicorn httpx pyogrio
!pip install -q -e .
print('\n✅ Environment ready!')

## 2. Verify GPU

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('⚠️  No GPU! Go to Runtime → Change runtime type → T4 GPU and restart.')

## 3. Download LEVIR-CD Dataset

Downloads from a public HuggingFace mirror (~470 MB). Takes ~30 seconds on Colab.

In [ ]:
import os, shutil
from pathlib import Path

%cd /content/OrbitalDelta
base = Path('data/raw/levir-cd')
base.mkdir(parents=True, exist_ok=True)
%cd /content/OrbitalDelta/data/raw/levir-cd

# Download from public HuggingFace mirror
print('Downloading images (~470 MB)...')
!wget -q --show-progress -O img.zip https://huggingface.co/datasets/sy2002123/levir-cd/resolve/main/img.zip
print('Downloading labels (~1 MB)...')
!wget -q --show-progress -O anno.zip https://huggingface.co/datasets/sy2002123/levir-cd/resolve/main/anno.zip

# Extract
print('Extracting...')
!unzip -qo img.zip
!unzip -qo anno.zip
!rm -f img.zip anno.zip

# Create A/, B/, label/ and sort files (images are named *_A.png / *_B.png)
base = Path('/content/OrbitalDelta/data/raw/levir-cd')
(base / 'A').mkdir(exist_ok=True)
(base / 'B').mkdir(exist_ok=True)
(base / 'label').mkdir(exist_ok=True)

img_dir, anno_dir = base / 'img', base / 'anno'

moved_a = moved_b = moved_l = 0
if img_dir.exists():
    for f in img_dir.iterdir():
        if f.suffix == '.png':
            if f.stem.endswith('_A'):
                shutil.move(str(f), str(base / 'A' / (f.stem[:-2] + '.png')))
                moved_a += 1
            elif f.stem.endswith('_B'):
                shutil.move(str(f), str(base / 'B' / (f.stem[:-2] + '.png')))
                moved_b += 1
    shutil.rmtree(img_dir)

if anno_dir.exists():
    for f in anno_dir.iterdir():
        if f.suffix == '.png':
            shutil.move(str(f), str(base / 'label' / f.name))
            moved_l += 1
    shutil.rmtree(anno_dir)

%cd /content/OrbitalDelta
print(f'✅ Done: {moved_a} A-images, {moved_b} B-images, {moved_l} labels')
!echo 'A/:' && ls data/raw/levir-cd/A | head -3
!echo 'B/:' && ls data/raw/levir-cd/B | head -3
!echo 'label/:' && ls data/raw/levir-cd/label | head -3

## 4. Preprocess Dataset

Crops images into 256×256 patches and creates train/val/test splits. Takes ~2 minutes.

In [ ]:
%cd /content/OrbitalDelta

!python -m src.data.preprocess \
    --input data/raw/levir-cd \
    --output data/processed/levir-cd \
    --crop-size 256

import os
print('\nPatch counts:')
for split in ['train', 'val', 'test']:
    n = len(os.listdir(f'data/processed/levir-cd/{split}/A'))
    print(f'  {split}: {n} image pairs')

## 5. Train the Model

Trains the Siamese U-Net. Best checkpoint saved to `checkpoints/best.pt`.  
**Expected time:** ~4–8 hours on T4 GPU with 50 epochs.

In [ ]:
%cd /content/OrbitalDelta

!python scripts/train.py \
    --config configs/train_levir.yaml \
    --epochs 50 \
    --batch-size 8

## 6. Evaluate the Model

Runs inference on the test set. Target: **F1 ≥ 0.88**.

In [ ]:
%cd /content/OrbitalDelta

!python scripts/evaluate.py \
    --checkpoint checkpoints/best.pt \
    --config configs/train_levir.yaml \
    --data-root data/processed/levir-cd/test

## 7. Download Trained Weights

In [ ]:
import os

if not os.path.exists('checkpoints/best.pt'):
    print('❌ checkpoints/best.pt not found. Make sure training completed!')
else:
    try:
        from google.colab import files
        files.download('checkpoints/best.pt')
        print('✅ Download started! Save best.pt to your local checkpoints/ folder.')
    except ImportError:
        print('⚠️  Not in Colab — copy checkpoints/best.pt manually.')